# Замер LinQ TPU — team_94

Сначала пример организаторов (resnet50) через наш клиент `linq_cloud.py`, затем замеры: задержка, пропускная
способность при нескольких задачах в полёте, результат → `bench_result.json` → журнал (вкладка «Система»).
Последний раздел — то же для `yolov8n-pose`, когда организаторы вернут скомпилированную программу.

Рядом с ноутбуком должны лежать `linq_cloud.py` и `linq_bench.py` (из папки `bench/` репозитория).

In [ ]:
import os, shutil, time, json
import numpy as np
from PIL import Image

LOGIN = 'hackathonXXXXX'                     # !!! свой логин Jupyter
PROGRAM_DIR = os.path.join('/auto/cache', LOGIN)
EXAMPLES = 'examples'                        # папка примера организаторов

RESNET = 'torch_resnet50_b1_o5_h1m.tpu'
IMG = os.path.join(EXAMPLES, 'ILSVRC2012_val_00000045.JPEG')

assert LOGIN != 'hackathonXXXXX', 'впишите свой логин'
os.makedirs(PROGRAM_DIR, exist_ok=True)
shutil.copy(os.path.join(EXAMPLES, RESNET), PROGRAM_DIR)   # воркер видит программы только из /auto/cache/<логин>
print(os.listdir(PROGRAM_DIR))

In [ ]:
from linq_cloud import LinqCloud, program_info
from linq_bench import Bench

info = program_info(os.path.join(PROGRAM_DIR, RESNET))
info   # имена, формы и типы входов/выходов — из metadata.json внутри .tpu

## 1. Проверка: resnet50 должен узнать ньюфаундленда (класс 257)

In [ ]:
def central_crop(image, percent=1.):
    w, h = image.size
    m = int(min(h, w) * percent)
    return image.crop(((w - m) // 2, (h - m) // 2, (w - m) // 2 + m, (h - m) // 2 + m))

def preprocess(image, hw=(224, 224)):
    image = central_crop(image.convert('RGB'), 0.85).resize(size=hw)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    arr = (np.asarray(image) / 255.0 - mean) / std
    return np.transpose(arr, (2, 0, 1)).astype(np.float32)[None]

IN_NAME, OUT_NAME = next(iter(info['inputs'])), next(iter(info['outputs']))   # 'input.1', '495'
x = preprocess(Image.open(IMG))

tpu = LinqCloud(os.path.join(PROGRAM_DIR, RESNET))
outs, meta = tpu.infer({IN_NAME: x})
cls = int(np.argmax(outs[OUT_NAME])) + 1
print(f'класс {cls} (ожидался 257)')
print(meta)   # rtt_ms — полный круг; tpu_ms — время на воркере; inference_time_raw — как прислал воркер
assert cls == 257

## 2. Задержка: 100 вызовов по одному

`tpu_ms` — время на воркере (из него — загрузка TPU и запас по камерам), «круг» — от отправки до ответа:
он ограничивает FPS одной камеры, если кадры отправляются строго по одному.

In [ ]:
b = Bench(node_id='linq-hackathon', node_name='LinQ TPU хакатона (облако)', tpu_cores=1)
b.measure('resnet50', lambda: tpu.infer({IN_NAME: x}), n=100, tpu_time=lambda r: r[1]['tpu_ms'])
b.table()

## 3. Пропускная способность: несколько задач в полёте

Так работают несколько камер: кадры уходят параллельно, и узел обслуживает их быстрее, чем «круг» по одному.

In [ ]:
rows = [tpu.throughput({IN_NAME: x}, n=64, in_flight=k) for k in (1, 2, 4, 8, 16)]
for r in rows:
    print(r)

## 4. Сохранить и загрузить в журнал

In [ ]:
b.save('bench_result.json')
json.dump({'throughput': {'resnet50': rows}}, open('bench_throughput.json', 'w'), ensure_ascii=False, indent=2)
print('скачайте bench_result.json и: python -m scripts.import_metrics bench_result.json --url http://localhost:8000')
# если из ВМ есть маршрут до журнала: b.push('http://<адрес журнала>:8000')

## 5. yolov8n-pose (и yolov8n), когда организаторы вернут программы

Отдаём им `core/models/yolov8n-pose_head.onnx` и `core/models/yolov8n_head.onnx` (голова без декодирования,
вход `images` 1×3×640×640 float32). Скомпилированные `.tpu` положите рядом с ноутбуком.
Для замера скорости подойдёт любой кадр; для проверки качества — кадр с людьми и сравнение с ONNX на ноутбуке.

In [ ]:
POSE = 'yolov8n-pose_head.tpu'                # имя файла от организаторов
FRAME = None                                   # путь к кадру с людьми (jpg) или None — шум для замера скорости

shutil.copy(POSE, PROGRAM_DIR)
pinfo = program_info(os.path.join(PROGRAM_DIR, POSE))
print(pinfo)
pin = next(iter(pinfo['inputs']))

def letterbox(path, size=640):
    im = Image.open(path).convert('RGB')
    r = min(size / im.width, size / im.height)
    im = im.resize((round(im.width * r), round(im.height * r)))
    canvas = Image.new('RGB', (size, size), (114, 114, 114))
    canvas.paste(im, ((size - im.width) // 2, (size - im.height) // 2))
    return (np.asarray(canvas, dtype=np.float32) / 255).transpose(2, 0, 1)[None]

px = letterbox(FRAME) if FRAME else np.random.rand(1, 3, 640, 640).astype(np.float32)
pose = LinqCloud(os.path.join(PROGRAM_DIR, POSE))
pouts, pmeta = pose.infer({pin: px})
print({k: v.shape for k, v in pouts.items()}, pmeta)   # голова: 3 масштаба × (64 рамки, 1 класс, 51 точка)

b.measure('yolo-pose', lambda: pose.infer({pin: px}), n=100, calls_per_frame=1, tpu_time=lambda r: r[1]['tpu_ms'])
b.table()
print([pose.throughput({pin: px}, n=48, in_flight=k) for k in (1, 4, 8)])
b.save('bench_result.json')